In [6]:
# pip install -U "modelscope==1.13.0" "datasets==2.14.6" "pyarrow==14.0.2" "numpy==1.26.4"

# from modelscope.msdatasets import MsDataset
# ds =  MsDataset.load('shareAI/shareAI-Llama3-DPO-zh-en-emoji', subset_name='default', split='train')



# 偏好对比（Preference Data）格式，用于 指令微调（SFT）和奖励模型（RM）训练 的高级数据集格式之一

from datasets import load_dataset
# ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized")

data_path = "/root/.cache/huggingface/datasets/rl2-dpo-datas-from-hf-ultrafeedback_binarized/default-28b1d4dfef2cc83d/0.0.0/8bb11242116d547c741b2e8a1f18598ffdd40a1d4f2a2872c7a28b697434bc96"

ds = load_dataset("/root/.cache/huggingface/datasets/rl2-dpo-datas-from-hf-ultrafeedback_binarized/default-28b1d4dfef2cc83d/0.0.0/8bb11242116d547c741b2e8a1f18598ffdd40a1d4f2a2872c7a28b697434bc96")

In [ ]:
# 查询数据概况，总体情况

print(ds)

DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected', 'prompt'],
        num_rows: 240
    })
    validation: Dataset({
        features: ['chosen', 'rejected', 'prompt'],
        num_rows: 60
    })
    test: Dataset({
        features: ['chosen', 'rejected', 'prompt'],
        num_rows: 300
    })
})


"\nDatasetDict({\n    train_prefs: Dataset({\n        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],\n        num_rows: 61135\n    })\n    train_sft: Dataset({\n        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],\n        num_rows: 61135\n    })\n    test_prefs: Dataset({\n        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],\n        num_rows: 2000\n    })\n    test_sft: Dataset({\n        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],\n        num_rows: 1000\n    })\n    train_gen: Dataset({\n        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],\n        num_rows: 61135\n    })\n    test_gen: Dataset({\n        features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rej

| 类型                       | 相关 split                    | 模型训练阶段        | 数据内容结构                           |
| ------------------------ | --------------------------- | ------------- | -------------------------------- |
| **SFT (监督微调)**           | `train_sft`, `test_sft`     | 教模型“如何正确回答”   | `messages` 对话格式（单个 assistant 回复） |
| **Preference (偏好/奖励模型)** | `train_prefs`, `test_prefs` | 教模型“哪个回答更好”   | 包含 `chosen` 和 `rejected` 两个回答    |
| **Generation (生成)**      | `train_gen`, `test_gen`     | 评估或微调“自由生成能力” | 模型自主生成任务，结构类似 SFT                |


1️⃣ train_sft / test_sft — 监督微调数据（SFT）

让模型学会「给出正确、有用的回答」。

特点：

结构：只包含一条完整对话（messages）

没有偏好信息（没有 chosen/rejected）

通常来源于人工或高质量模型示例

常用于“第1阶段：监督微调（Supervised Fine-Tuning）”

示例结构：

```shell
{
  "prompt": "Explain why the sky is blue.",
  "messages": [
    {"role": "user", "content": "Why is the sky blue?"},
    {"role": "assistant", "content": "The sky appears blue because..."}
  ]
}
```


2️⃣ train_prefs / test_prefs — 偏好比较数据（Preference Data）

让模型学会「哪种回答更受人类喜爱」。

特点：

包含两种回答：

✅ chosen（好回答）

❌ rejected（坏回答）

常用于训练 Reward Model 或 DPO（Direct Preference Optimization）

是“第2阶段：人类反馈训练（RLHF 或 DPO）”的主要数据

示例结构：

```shell
{
  "prompt": "How can I develop a habit of drawing daily?",
  "chosen": [{"role": "assistant", "content": "Start small, draw 10 min/day..."}],
  "rejected": [{"role": "assistant", "content": "I cannot help you form habits."}],
  "score_chosen": 8.5,
  "score_rejected": 5.0
}
```


3️⃣ train_gen / test_gen — 生成任务数据（Generation Data）

用于模型的“自由生成”和“内容评估”。

特点：

结构类似于 train_sft

通常没有 rejected

用于“第3阶段：生成能力调优”或“自动评估生成任务”

有时是从偏好数据中提取出的单个优质回答

示例结构：

```shell
{
  "prompt": "Write a poem about AI.",
  "messages": [
    {"role": "user", "content": "Write a poem about AI"},
    {"role": "assistant", "content": "In circuits deep, where minds awake..."}
  ]
}
```

In [ ]:
import random
from pprint import pprint
from datasets import load_dataset

# 加载数据集
ds = load_dataset(data_path)

# 选择一个子集（split） 
""" 
这个 split 的每一条数据其实描述了一次 人机对话 (messages)，
并提供了两个候选回答：

一个是「chosen」✅（人类偏好的回答）

一个是「rejected」❌（人类不喜欢的回答）

这类数据常用于训练：

奖励模型（Reward Model）

偏好优化（DPO、PPO）

对齐模型（Alignment / RLHF）
"""
subset = ds["train_prefs"]

# 打印数据条数
print(f"Total samples in train_prefs: {len(subset)}\n")

# 查看前几条数据（比如前 3 条）
for i in range(3):
    print(f"===== Sample {i} =====")
    pprint(subset[i])
    print("\n")

# 字段解释
"""
| 字段               | 类型           | 含义                                        |
| ---------------- | ------------ | ----------------------------------------- |
| `prompt`         | `str`        | 原始用户输入（指令）                                |
| `prompt_id`      | `str`        | 唯一标识符（哈希值）                                |
| `messages`       | `list[dict]` | 完整对话（包括 user 和 assistant）——通常是 chosen 的副本 |
| `chosen`         | `list[dict]` | 被认为“更好”的完整对话（包含角色 + 回复）                   |
| `rejected`       | `list[dict]` | 被认为“较差”的完整对话                              |
| `score_chosen`   | `float`      | 偏好得分（模型或人工打分）                             |
| `score_rejected` | `float`      | 反面样本得分                                    |


+---------------------------+
| prompt: "how can i draw?" |
+---------------------------+
| chosen: [好回答✅]        |
| rejected: [差回答❌]       |
| messages: [完整对话]       |
| score_chosen: 8.5         |
| score_rejected: 8.5       |
+---------------------------+
"""

# output demo

"""
Downloading readme: 6.53kB [00:00, 5.88MB/s]
Total samples in train_prefs: 61135

===== Sample 0 =====
{'chosen': [{'content': 'how can i develop a habit of drawing daily',
             'role': 'user'},
            {'content': 'Developing a daily habit of drawing can be '
                        'challenging but with consistent practice and a few '
                        'tips, it can become an enjoyable and rewarding part '
                        'of your daily routine. Here are some strategies to '
                        'help you develop the habit of drawing daily:\n'
                        '\n'
                        '1. Set a specific time: Allocate a specific time of '
                        'the day to draw. It could be in the morning, '
                        'afternoon, or evening. Make drawing a part of your '
                        'daily routine.\n'
                        '2. Set a specific duration: Determine the amount of '
                        'time you want to spend on drawing each day. It can be '
                        'as little as 10 minutes or as long as an hour. Be '
                        'consistent with the duration to help build the '
                        'habit.\n'
                        "3. Start small and simple: Don't try to create a "
                        'masterpiece every day, start with simple and '
                        'easy-to-do sketches. Focus on improving your skills '
                        'gradually.\n'
                        '4. Use a variety of tools and mediums: Experiment '
...
 'score_chosen': 6.0,
 'score_rejected': 3.0}
"""

In [9]:
# convert_ultrafeedback_to_ppo.py  
from datasets import load_dataset  
import json  
import random  
from typing import List, Dict, Optional  
  
def convert_ultrafeedback_to_ppo(  
    output_train: str = "ppo_train.json",  
    output_test: Optional[str] = None,  
    test_ratio: float = 0.1,  
    num_samples: Optional[int] = None,  
    seed: int = 42  
):  
    """  
    将 ultrafeedback_binarized 数据集转换为 PPO 训练格式  
      
    Args:  
        output_train: 训练数据输出文件路径  
        output_test: 测试数据输出文件路径，None表示不划分测试集  
        test_ratio: 测试集比例  
        num_samples: 限制样本数量，None表示使用全部数据  
        seed: 随机种子  
    """  
      
    # 设置随机种子  
    random.seed(seed)  
      
    # 加载数据集  
    print("Loading ultrafeedback_binarized dataset...")  
    dataset = load_dataset(data_path, split="train")  
      
    if num_samples:  
        dataset = dataset.select(range(min(num_samples, len(dataset))))  
      
    ppo_data = []  
      
    print("Converting data...")  
    for idx, item in enumerate(dataset):  
        # 处理 chosen 字段（messages格式）  
        if isinstance(item["chosen"], list):  
            messages = item["chosen"]  
            prompt_messages = [msg for msg in messages if msg["role"] == "user"]  
              
            if prompt_messages:  
                # 取最后一个用户消息作为prompt  
                prompt = prompt_messages[-1]["content"]  
                  
                # 提取assistant的回答作为参考答案  
                assistant_messages = [msg for msg in messages if msg["role"] == "assistant"]  
                reference_answer = assistant_messages[-1]["content"] if assistant_messages else ""  
        else:  
            # 处理可能的字符串格式（虽然当前数据集是messages格式）  
            prompt = item.get("prompt", "")  
            reference_answer = item.get("chosen", "")  
          
        ppo_item = {  
            "prompt": prompt,  
            "extra_info": {  
                "reference_answer": reference_answer,  
                "original_id": idx  # 保留原始索引  
            }  
        }  
        ppo_data.append(ppo_item)  
      
    print(f"Converted {len(ppo_data)} samples")  
      
    # 划分数据  
    if output_test is not None:  
        print(f"Splitting data with test ratio: {test_ratio}")  
        random.shuffle(ppo_data)  
          
        split_idx = int(len(ppo_data) * (1 - test_ratio))  
        train_data = ppo_data[:split_idx]  
        test_data = ppo_data[split_idx:]  
          
        print(f"Train samples: {len(train_data)}")  
        print(f"Test samples: {len(test_data)}")  
          
        # 保存训练数据  
        print(f"Saving train data to {output_train}...")  
        with open(output_train, "w", encoding="utf-8") as f:  
            json.dump(train_data, f, ensure_ascii=False, indent=2)  
          
        # 保存测试数据  
        print(f"Saving test data to {output_test}...")  
        with open(output_test, "w", encoding="utf-8") as f:  
            json.dump(test_data, f, ensure_ascii=False, indent=2)  
    else:  
        # 不划分，全部作为训练数据  
        print(f"Saving all data to {output_train}...")  
        with open(output_train, "w", encoding="utf-8") as f:  
            json.dump(ppo_data, f, ensure_ascii=False, indent=2)  
      
    print("Conversion completed!")  
    return ppo_data  
  
if __name__ == "__main__":  
    # 使用示例1：只生成训练数据（不划分）  
    # convert_ultrafeedback_to_ppo(  
    #     output_train="ppo_train.json",  
    #     output_test=None,  # 不划分测试集  
    #     num_samples=1000   # 先用1000条测试  
    # )  
      
    # 使用示例2：划分训练和测试数据  
    convert_ultrafeedback_to_ppo(  
        output_train="ppo_train-900.json",  
        output_test="ppo_test-100.json",  
        test_ratio=0.1,  
        num_samples=5000  
    )

Loading ultrafeedback_binarized dataset...
Converting data...
Converted 240 samples
Splitting data with test ratio: 0.1
Train samples: 216
Test samples: 24
Saving train data to ppo_train-900.json...
Saving test data to ppo_test-100.json...
Conversion completed!
